# Explore Results for a Specific Run

This notebook loads and explores results from a specific environment/method/num_episodes combination.

For example: all batches for DQN on CartPole with 50 episodes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Configuration

Set the experiment parameters you want to explore:

In [19]:
# Configuration - modify these to explore different runs
experiment_id = "humanoid_baseline"
method = "dqn"  # e.g., "monte_carlo", "dqn", "least_squares_mc", "least_squares_td"
num_episodes = 400  # number of episodes used for training

# Paths
base_path = Path("/scratch/dc3430/pooling_eval/experiments") / experiment_id
results_path = base_path / "results" / method / str(num_episodes)
predictions_file = results_path / "predictions.parquet"
metadata_file = results_path / "predictions_metadata.json"

print(f"Loading results from: {results_path}")
print(f"Predictions file exists: {predictions_file.exists()}")
print(f"Metadata file exists: {metadata_file.exists()}")

Loading results from: /scratch/dc3430/pooling_eval/experiments/humanoid_baseline/results/dqn/400
Predictions file exists: True
Metadata file exists: True


## Load Data

In [20]:
# Load predictions
df = pd.read_parquet(predictions_file)
print(f"Loaded {len(df)} predictions")
print(f"\nColumns: {list(df.columns)}")
print(f"\nShape: {df.shape}")
df.head()

Loaded 3117360 predictions

Columns: ['state_idx', 'episode_idx', 'batch_name', 'predicted_value']

Shape: (3117360, 4)


,state_idx,episode_idx,batch_name,predicted_value
0,0,0,0,9144.195312
1,1,0,0,6660.422852
2,2,0,0,5387.819336
3,3,0,0,7981.313477
4,4,0,0,7367.719727


In [21]:
# Load metadata
with open(metadata_file, 'r') as f:
    metadata = json.load(f)

print("Metadata:")
for key, value in metadata.items():
    if key not in ['estimator_config', 'network_config']:
        print(f"  {key}: {value}")

print("\nEstimator Config:")
for key, value in metadata['estimator_config'].items():
    print(f"  {key}: {value}")

print("\nNetwork Config:")
for key, value in metadata['network_config'].items():
    print(f"  {key}: {value}")

Metadata:
  experiment_id: humanoid_baseline
  method: dqn
  n_episodes: 400
  n_batches: 80
  n_states: 38967
  n_eval_episodes: 200
  created_at: 2026-01-19T14:11:34.871883
  policy_environment: Humanoid-v4
  policy_algorithm: PPO
  policy_seed: 42
  policy_learning_rate: 3e-05
  policy_gamma: 0.95
  policy_total_timesteps: 80000000
  policy_average_reward: 2227.08303148
  data_seed: 42
  policy_display_name: None

Estimator Config:
  name: dqn
  type: dqn
  learning_rate: 0.02
  n_initializations: 1
  max_epochs: None
  target_update_rate: 0.04

Network Config:
  hidden_sizes: [256, 256, 128]
  activation: relu


## Basic Statistics

In [22]:
# Summary statistics
print("Summary Statistics:")
print(df.describe())

Summary Statistics:
          state_idx   episode_idx  predicted_value
count  3.117360e+06  3.117360e+06     3.117360e+06
mean   1.948300e+04  1.066745e+02     8.959732e+03
std    1.124881e+04  5.438222e+01     5.349090e+03
min    0.000000e+00  0.000000e+00    -1.369745e+05
25%    9.741000e+03  6.300000e+01     4.799425e+03
50%    1.948300e+04  1.080000e+02     9.218206e+03
75%    2.922500e+04  1.520000e+02     1.281677e+04
max    3.896600e+04  1.990000e+02     1.389133e+05


## Analysis by Batch

In [23]:
# Statistics per batch
batch_stats = df.groupby(['state_idx']).agg({
    'predicted_value': ['mean', 'std'],
}).round(4).reset_index()
batch_stats.columns = ['state_idx', 'mean_predicted_value', 'std_predicted_value']
print("\nStatistics by Batch:")
batch_stats


Statistics by Batch:


,state_idx,mean_predicted_value,std_predicted_value
0,0,7258.472168,2275.2540
1,1,5011.114258,3686.8595
2,2,4429.733398,3468.1396
3,3,5881.847168,3458.3251
4,4,5417.506348,3884.4324
...,...,...,...
38962,38962,84.748100,295.7382
38963,38963,57.230301,290.2003
38964,38964,86.484596,273.1313
38965,38965,43.299099,184.3662


In [28]:
df['state_position_in_episode'] = df.groupby(['episode_idx', 'batch_name']).cumcount()

In [38]:
df[df['state_idx'] == 4350]

,state_idx,episode_idx,batch_name,predicted_value,state_position_in_episode
4350,4350,32,0,33.914433,154
43317,4350,32,1,10.136691,154
82284,4350,32,2,12.444998,154
121251,4350,32,3,3.542138,154
160218,4350,32,4,95.244614,154
...,...,...,...,...,...
2926875,4350,32,75,2.073720,154
2965842,4350,32,76,9.860098,154
3004809,4350,32,77,14.061638,154
3043776,4350,32,78,10.247486,154


In [24]:
batch_stats['ratio'] = batch_stats['std_predicted_value'] / batch_stats['mean_predicted_value']
batch_stats.sort_values(by='ratio', ascending=False).head(10)

,state_idx,mean_predicted_value,std_predicted_value,ratio
4350,4350,0.2314,271.2551,1172.234668
35502,35502,0.4640,331.0533,713.476960
36068,36068,0.5146,350.3094,680.741186
29296,29296,0.6666,264.0906,396.175524
17472,17472,1.4546,425.9734,292.845736
4349,4349,1.1605,329.7899,284.179135
12080,12080,0.7607,212.0903,278.809391
27104,27104,0.9694,227.5819,234.765734
31301,31301,1.1824,251.4568,212.666443
17109,17109,0.8356,153.9266,184.210862


In [41]:
import plotly.express as px
pd.options.plotting.backend = 'plotly'


In [44]:
episode = 32
sorted_state_idx = df[df['episode_idx'] == episode].sort_values(by='state_position_in_episode')['state_idx'].unique()
episode_stats = batch_stats[batch_stats['state_idx'].isin(sorted_state_idx)].sort_values(by='state_idx')
episode_stats.reset_index(drop=True, inplace=True)
episode_stats['mean_predicted_value'].plot()
episode_stats['std_predicted_value'].plot()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'variable=std_predicted_value<br>index=%{x}<br>value=%{y}<extra></extra>',
              'legendgroup': 'std_predicted_value',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': 'std_predicted_value',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'CRAJIAkwCUAJUAlgCXAJgAmQCaAA=='),
                    'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('bjSAtyDqoUCM22gAT76tQO0NvjAZxq' ... 'Fc/3NAxLEubqOcdEBb07zjFPRwQA=='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'margin': {'t': 60},
               'template': '...',
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'index'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'value'}}}
})

## Visualizations

In [ ]:
# Plot 1: Predicted vs True Values (scatter plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall scatter
axes[0].scatter(df['true_value'], df['predicted_value'], alpha=0.3, s=10)
axes[0].plot([df['true_value'].min(), df['true_value'].max()], 
             [df['true_value'].min(), df['true_value'].max()], 
             'r--', label='Perfect prediction')
axes[0].set_xlabel('True Value')
axes[0].set_ylabel('Predicted Value')
axes[0].set_title('Predicted vs True Values (All Batches)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# By batch
for batch_idx in df['batch_idx'].unique():
    batch_df = df[df['batch_idx'] == batch_idx]
    axes[1].scatter(batch_df['true_value'], batch_df['predicted_value'], 
                   alpha=0.5, s=10, label=f'Batch {batch_idx}')
axes[1].plot([df['true_value'].min(), df['true_value'].max()], 
            [df['true_value'].min(), df['true_value'].max()], 
            'k--', label='Perfect prediction', linewidth=2)
axes[1].set_xlabel('True Value')
axes[1].set_ylabel('Predicted Value')
axes[1].set_title('Predicted vs True Values (By Batch)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Distribution of prediction errors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall histogram
axes[0].hist(df['prediction_error'], bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
axes[0].axvline(df['prediction_error'].mean(), color='g', linestyle='--', 
               linewidth=2, label=f'Mean error: {df["prediction_error"].mean():.3f}')
axes[0].set_xlabel('Prediction Error')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Prediction Errors')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# By batch (boxplot)
df.boxplot(column='prediction_error', by='batch_idx', ax=axes[1])
axes[1].axhline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
axes[1].set_xlabel('Batch Index')
axes[1].set_ylabel('Prediction Error')
axes[1].set_title('Prediction Error Distribution by Batch')
axes[1].legend()
plt.suptitle('')  # Remove the default boxplot title

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Error metrics by batch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

batch_indices = sorted(df['batch_idx'].unique())

# MSE by batch
mse_by_batch = [df[df['batch_idx'] == idx]['squared_error'].mean() for idx in batch_indices]
axes[0].bar(batch_indices, mse_by_batch, alpha=0.7, edgecolor='black')
axes[0].axhline(np.mean(mse_by_batch), color='r', linestyle='--', 
               linewidth=2, label=f'Mean MSE: {np.mean(mse_by_batch):.4f}')
axes[0].set_xlabel('Batch Index')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('MSE by Batch')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# MAE by batch
mae_by_batch = [df[df['batch_idx'] == idx]['absolute_error'].mean() for idx in batch_indices]
axes[1].bar(batch_indices, mae_by_batch, alpha=0.7, edgecolor='black', color='orange')
axes[1].axhline(np.mean(mae_by_batch), color='r', linestyle='--', 
               linewidth=2, label=f'Mean MAE: {np.mean(mae_by_batch):.4f}')
axes[1].set_xlabel('Batch Index')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].set_title('MAE by Batch')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Value distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of true values
axes[0].hist(df['true_value'], bins=50, alpha=0.7, label='True values', edgecolor='black')
axes[0].axvline(df['true_value'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f'Mean: {df["true_value"].mean():.3f}')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of True Values')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution of predicted values
axes[1].hist(df['predicted_value'], bins=50, alpha=0.7, label='Predicted values', 
            edgecolor='black', color='orange')
axes[1].axvline(df['predicted_value'].mean(), color='r', linestyle='--', 
               linewidth=2, label=f'Mean: {df["predicted_value"].mean():.3f}')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Predicted Values')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Episode-Level Analysis

In [ ]:
# Statistics per episode
episode_stats = df.groupby(['batch_idx', 'episode_idx']).agg({
    'predicted_value': ['mean', 'std'],
    'true_value': ['mean', 'std'],
    'squared_error': 'mean',
    'absolute_error': 'mean',
    'state_idx': 'count'  # number of states in episode
}).round(4)

episode_stats.columns = ['_'.join(col).strip() for col in episode_stats.columns.values]
episode_stats.rename(columns={'state_idx_count': 'n_states'}, inplace=True)

print("\nEpisode Statistics (first 10 episodes):")
episode_stats.head(10)

In [ ]:
# Plot: MSE vs episode length
fig, ax = plt.subplots(figsize=(10, 6))

for batch_idx in df['batch_idx'].unique():
    batch_episode_stats = episode_stats.loc[batch_idx]
    ax.scatter(batch_episode_stats['n_states'], 
              batch_episode_stats['squared_error_mean'],
              alpha=0.6, s=50, label=f'Batch {batch_idx}')

ax.set_xlabel('Episode Length (number of states)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('MSE vs Episode Length')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Custom Analysis

Use the cells below for your own exploratory analysis:

In [ ]:
# Your custom analysis here
